In [40]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [41]:
train = np.loadtxt('./data/digit/train.csv', delimiter=',', skiprows=1)
test = np.loadtxt('./data/digit/test.csv', delimiter=',', skiprows=1)

In [42]:
train_label = train[:, 0]
train_img = np.resize(train[:, 1:], (train.shape[0], 28, 28))
test_img = np.resize(test, (test.shape[0], 28, 28))

In [43]:
train_img.shape

(42000, 28, 28)

In [44]:
train_sobel_x = np.zeros_like(train_img)
train_sobel_y = np.zeros_like(train_img)
for i in range(len(train_img)):
    train_sobel_x[i] = cv2.Sobel(train_img[i], cv2.CV_64F, 1, 0, ksize=3)
    train_sobel_y[i] = cv2.Sobel(train_img[i], cv2.CV_64F, 0, 1, ksize=3)

In [45]:
test_sobel_x = np.zeros_like(test_img)
test_sobel_y = np.zeros_like(test_img)
for i in range(len(test_img)):
    test_sobel_x[i] = cv2.Sobel(test_img[i], cv2.CV_64F, 1, 0, ksize=3)
    test_sobel_y[i] = cv2.Sobel(test_img[i], cv2.CV_64F, 0, 1, ksize=3)

In [46]:
train_g, train_theta = cv2.cartToPolar(train_sobel_x, train_sobel_y)

In [47]:
test_g, test_theta = cv2.cartToPolar(test_sobel_x, test_sobel_y)

In [48]:
train_hist = np.zeros((len(train_img), 16))
for i in range(len(train_img)):
    hist, _ = np.histogram(train_theta[i], bins=16, range=(0., 2 * np.pi), weights=train_g[i])
    train_hist[i] = hist

In [49]:
test_hist = np.zeros((len(test_img), 16))
for i in range(len(test_img)):
    hist, _ = np.histogram(test_theta[i], bins=16, range=(0., 2 * np.pi), weights=test_g[i])
    test_hist[i] = hist

In [50]:
X_train, X_val, y_train, y_val = train_test_split(train_hist, train_label, test_size=0.2, random_state=42)

In [51]:
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn.fit(X_train, y_train)

,n_neighbors,5
,weights,'distance'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [52]:
y_pred = knn.predict(X_val)
print('Accuracy:', accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

Accuracy: 0.6567857142857143
              precision    recall  f1-score   support

         0.0       0.68      0.64      0.66       816
         1.0       0.96      0.97      0.96       909
         2.0       0.45      0.38      0.41       846
         3.0       0.63      0.71      0.67       937
         4.0       0.75      0.65      0.70       839
         5.0       0.61      0.66      0.63       702
         6.0       0.46      0.47      0.46       785
         7.0       0.82      0.75      0.78       893
         8.0       0.57      0.59      0.58       835
         9.0       0.62      0.70      0.66       838

    accuracy                           0.66      8400
   macro avg       0.65      0.65      0.65      8400
weighted avg       0.66      0.66      0.66      8400



In [53]:
y_test_pred = knn.predict(test_hist)

In [54]:
with open('submission.csv', 'w') as dst:
    dst.write('ImageId,Label\n')
    for i, p in enumerate(y_test_pred, 1):
        dst.write(f'{i},{int(p)}\n')